# Script 2 Pilot 8 Participant EEG Semi-Automatic Batch Processing

This notebook is independent from the single-recording QC notebook. It reproduces the same automatic processing across the 24 pilot recordings (IC01–IC05 and N01–N03, three scenes each).

**Relationship between the two notebooks**

- Notebook 1 explains one recording and makes the automatic QC logic visible.
- Notebook 2 applies the same fixed logic to all eight pilot participants.
- Notebook 2 does not import or require Notebook 1.

**Semi-automatic workflow**

- First run: automatic QC, QC plots, review queue and provisional EEG results are generated.
- Human decisions are stored only in `pilot8_manual_review_decisions.csv`.
- Blank means accept the automatic result; `keep` restores an automatically rejected epoch; `reject` excludes an epoch.
- After editing the CSV, restart the kernel and run all cells again.
- No participant-specific decision is hard-coded in the notebook.


## Imports and plotting style


In [1]:

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import trapezoid
from IPython.display import display

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False




## 1. User Settings


In [2]:
# =========================================================
# 1. USER SETTINGS
# =========================================================

# Pilot-data directory.
DATA_DIR = Path(r"C:\Users\Harvey McWalter\Desktop\EEG_Data")
OUTPUT_DIR = DATA_DIR / "Pilot8_EEG_SemiAutomatic_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 500
RIGHT_EEG_COL = "CH1"
LEFT_EEG_COL = "CH2"
ECG_COL = "CH3"

SCENE_NAMES = {
    "BG": "Blue-Green",
    "G": "Green",
    "GR": "Grey",
}
SCENE_SORT_ORDER = {"BG": 0, "G": 1, "GR": 2}

EEG_BAND = (1.0, 30.0)
THETA_BAND = (4.0, 8.0)
ALPHA_BAND = (8.0, 13.0)
BETA_BAND = (13.0, 30.0)
TOTAL_POWER_BAND = (4.0, 30.0)

EPOCH_SECONDS = 4.0
FILTER_ORDER = 4
START_TRIM_SECONDS = 15.0
END_TRIM_SECONDS = 0.0
V_TO_UV = 1_000_000
NUMBER_OF_MADS = 6.0
WELCH_WINDOW_SECONDS = 2.0

# Set to True to automatically find all files matching *_EEG.csv in DATA_DIR.
AUTO_DISCOVER_FILES = False

# If AUTO_DISCOVER_FILES = False, edit this list manually.
EXPECTED_FILES = [
    "IC01_BG_EEG.csv",
    "IC01_G_EEG.csv",
    "IC01_GR_EEG.csv",
    "IC02_BG_EEG.csv",
    "IC02_G_EEG.csv",
    "IC02_GR_EEG.csv",
    "IC03_BG_EEG.csv",
    "IC03_G_EEG.csv",
    "IC03_GR_EEG.csv",
    "IC04_BG_EEG.csv",
    "IC04_G_EEG.csv",
    "IC04_GR_EEG.csv",
    "IC05_BG_EEG.csv",
    "IC05_G_EEG.csv",
    "IC05_GR_EEG.csv",
    "N01_BG_EEG.csv",
    "N01_G_EEG.csv",
    "N01_GR_EEG.csv",
    "N02_BG_EEG.csv",
    "N02_G_EEG.csv",
    "N02_GR_EEG.csv",
    "N03_BG_EEG.csv",
    "N03_G_EEG.csv",
    "N03_GR_EEG.csv",
]

# Human decisions are stored outside the code.
# Blank = accept the automatic result; keep = restore; reject = exclude.
DECISION_FILE = OUTPUT_DIR / "pilot8_manual_review_decisions.csv"

def load_manual_decisions(decision_file):
    required_columns = [
        "participant", "scene_code", "channel", "epoch", "manual_decision"
    ]

    if not decision_file.exists():
        return pd.DataFrame(columns=required_columns + ["review_note", "reviewer"])

    table = pd.read_csv(decision_file, keep_default_na=False)
    missing = [column for column in required_columns if column not in table.columns]
    if missing:
        raise KeyError(f"Decision file is missing columns: {missing}")

    table["participant"] = table["participant"].astype(str).str.strip().str.upper()
    table["scene_code"] = table["scene_code"].astype(str).str.strip().str.upper()
    table["channel"] = table["channel"].astype(str).str.strip().str.lower()
    table["epoch"] = pd.to_numeric(table["epoch"], errors="raise").astype(int)
    table["manual_decision"] = (
        table["manual_decision"].astype(str).str.strip().str.lower()
    )

    invalid = ~table["manual_decision"].isin(["", "keep", "reject"])
    if invalid.any():
        display(table.loc[invalid])
        raise ValueError("manual_decision must be blank, keep or reject.")

    duplicate = table.duplicated(
        subset=["participant", "scene_code", "channel", "epoch"],
        keep=False,
    )
    if duplicate.any():
        display(table.loc[duplicate])
        raise ValueError("Duplicate manual-decision keys were found.")

    for optional_column in ["review_note", "reviewer"]:
        if optional_column not in table.columns:
            table[optional_column] = ""

    return table


MANUAL_DECISIONS = load_manual_decisions(DECISION_FILE)
SAVE_QC_PLOTS = True





## 2. Common Functions


In [3]:
# =========================================================
# 2. COMMON FUNCTIONS 
# =========================================================

def parse_filename(file_path: Path) -> dict:
    pattern = re.compile(
        r"^(?P<participant>[A-Za-z]+\d+)_(?P<scene>BG|GR|G)_EEG$",
        flags=re.IGNORECASE,
    )
    match = pattern.fullmatch(file_path.stem)
    if match is None:
        raise ValueError(
            f"Filename does not match the expected format: {file_path.name}"
        )

    participant = match.group("participant").upper()
    scene_code = match.group("scene").upper()
    return {
        "participant": participant,
        "scene_code": scene_code,
        "scene_name": SCENE_NAMES[scene_code],
    }


def discover_input_files(data_dir: Path):
    pattern = re.compile(
        r"^(?P<participant>[A-Za-z]+\d+)_(?P<scene>BG|GR|G)_EEG\.csv$",
        flags=re.IGNORECASE,
    )
    files = []
    for path in data_dir.glob("*_EEG.csv"):
        if pattern.fullmatch(path.name):
            files.append(path)

    if not files:
        raise FileNotFoundError(
            f"No matching *_EEG.csv files were found in {data_dir}"
        )

    def sort_key(path: Path):
        info = parse_filename(path)
        return (
            info["participant"],
            SCENE_SORT_ORDER[info["scene_code"]],
        )

    return sorted(files, key=sort_key)


def load_recording(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()

    required_columns = [RIGHT_EEG_COL, LEFT_EEG_COL, ECG_COL]
    missing_columns = [c for c in required_columns if c not in df.columns]
    if missing_columns:
        raise KeyError(
            f"{file_path.name}: missing columns {missing_columns}. "
            f"Available columns: {df.columns.tolist()}"
        )

    for column in required_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    missing_values = df[required_columns].isna().sum()
    if missing_values.sum() > 0:
        raise ValueError(
            f"{file_path.name}: NaN or non-numeric samples were found:\n"
            f"{missing_values}"
        )

    return df


def bandpass_filter(x, fs, lowcut, highcut, order=4):
    x = np.asarray(x, dtype=float)
    if x.ndim != 1:
        raise ValueError("Input signal must be one-dimensional.")
    if not np.all(np.isfinite(x)):
        raise ValueError("Signal contains NaN or infinite values.")

    sos = signal.butter(
        N=order,
        Wn=[lowcut, highcut],
        btype="bandpass",
        fs=fs,
        output="sos",
    )
    return signal.sosfiltfilt(sos, x)


def create_epochs(x, fs, epoch_seconds):
    x = np.asarray(x, dtype=float)
    samples_per_epoch = int(round(fs * epoch_seconds))
    number_of_epochs = len(x) // samples_per_epoch
    if number_of_epochs < 1:
        raise ValueError("Recording is shorter than one complete epoch.")

    trimmed_length = number_of_epochs * samples_per_epoch
    return x[:trimmed_length].reshape(number_of_epochs, samples_per_epoch)


def robust_upper_threshold(values, number_of_mads=6.0):
    values = np.asarray(values, dtype=float)
    median_value = np.median(values)
    mad = np.median(np.abs(values - median_value))
    robust_sd = 1.4826 * mad

    if robust_sd == 0:
        return median_value * 3 if median_value > 0 else np.inf

    threshold = median_value + number_of_mads * robust_sd
    return max(threshold, median_value * 3)


def assess_epoch_quality(epochs, epoch_seconds, number_of_mads=6.0):
    epochs = np.asarray(epochs, dtype=float)
    if epochs.ndim != 2:
        raise ValueError("Epoch data must be a 2D array.")

    peak_to_peak = np.ptp(epochs, axis=1)
    maximum_step = np.max(np.abs(np.diff(epochs, axis=1)), axis=1)

    ptp_threshold = robust_upper_threshold(peak_to_peak, number_of_mads)
    step_threshold = robust_upper_threshold(maximum_step, number_of_mads)

    finite_mask = np.all(np.isfinite(epochs), axis=1)
    non_flat_mask = peak_to_peak > 0
    ptp_clean_mask = peak_to_peak <= ptp_threshold
    step_clean_mask = maximum_step <= step_threshold

    clean_mask = finite_mask & non_flat_mask & ptp_clean_mask & step_clean_mask

    quality_table = pd.DataFrame({
        "epoch": np.arange(len(epochs)),
        "start_time_s": np.arange(len(epochs)) * epoch_seconds,
        "end_time_s": (np.arange(len(epochs)) + 1) * epoch_seconds,
        "peak_to_peak_uv": peak_to_peak,
        "maximum_step_uv": maximum_step,
        "failed_ptp": ~ptp_clean_mask,
        "failed_step": ~step_clean_mask,
        "clean": clean_mask,
    })

    return {
        "quality_table": quality_table,
        "clean_mask": clean_mask,
        "ptp_threshold_uv": ptp_threshold,
        "step_threshold_uv": step_threshold,
    }


def integrate_band_power(psd_epochs, frequencies, low_frequency, high_frequency):
    band_mask = (frequencies >= low_frequency) & (frequencies <= high_frequency)
    return trapezoid(
        psd_epochs[:, band_mask],
        frequencies[band_mask],
        axis=1,
    )


def classify_epoch_status(
    quality_table,
    participant,
    scene_code,
    channel,
    decision_table,
):
    table = quality_table.copy()

    decision_subset = decision_table.loc[
        (decision_table["participant"] == participant)
        & (decision_table["scene_code"] == scene_code)
        & (decision_table["channel"] == channel),
        ["epoch", "manual_decision"],
    ].copy()

    decision_lookup = dict(
        zip(decision_subset["epoch"], decision_subset["manual_decision"])
    )

    auto_reason = []
    status = []
    final_keep = []
    manual_decision_values = []

    for _, row in table.iterrows():
        epoch = int(row["epoch"])
        failed_ptp = bool(row["failed_ptp"])
        failed_step = bool(row["failed_step"])

        if failed_ptp and failed_step:
            automatic_reason = "both"
            automatic_status = "auto_reject_both"
            automatic_keep = False
        elif failed_ptp:
            automatic_reason = "ptp_only"
            automatic_status = "auto_reject_ptp"
            automatic_keep = False
        elif failed_step:
            automatic_reason = "step_only"
            automatic_status = "auto_reject_step"
            automatic_keep = False
        else:
            automatic_reason = "kept"
            automatic_status = "clean"
            automatic_keep = True

        decision = decision_lookup.get(epoch, "")

        if decision == "keep":
            final_status = "manual_restored"
            keep_value = True
        elif decision == "reject":
            final_status = "manual_bad"
            keep_value = False
        else:
            final_status = automatic_status
            keep_value = automatic_keep

        auto_reason.append(automatic_reason)
        status.append(final_status)
        final_keep.append(keep_value)
        manual_decision_values.append(decision)

    table["auto_reason"] = auto_reason
    table["manual_decision"] = manual_decision_values
    table["status"] = status
    table["final_keep"] = final_keep
    return table


def save_qc_plot(file_path, metadata, right_status_table, left_status_table, right_quality, left_quality):
    status_styles = {
        "clean": {
            "color": "#4c78a8",
            "marker": "o",
            "label": "Kept",
            "size": 22,
            "alpha": 0.8,
        },
        "auto_reject_ptp": {
            "color": "#e15759",
            "marker": "x",
            "label": "Auto reject: peak-to-peak only",
            "size": 60,
            "alpha": 1.0,
        },
        "auto_reject_step": {
            "color": "#f28e2b",
            "marker": "s",
            "label": "Auto reject: maximum-step only",
            "size": 42,
            "alpha": 1.0,
        },
        "auto_reject_both": {
            "color": "#b07aa1",
            "marker": "D",
            "label": "Auto reject: both rules",
            "size": 46,
            "alpha": 1.0,
        },
        "manual_bad": {
            "color": "#000000",
            "marker": "P",
            "label": "Manual reject",
            "size": 55,
            "alpha": 1.0,
        },
        "manual_restored": {
            "color": "#59a14f",
            "marker": "*",
            "label": "Manually restored",
            "size": 95,
            "alpha": 1.0,
        },
    }

    fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)

    panel_items = [
        (axes[0, 0], right_status_table, "peak_to_peak_uv", right_quality["ptp_threshold_uv"], "Right-ear peak-to-peak", "Right PtP threshold"),
        (axes[0, 1], right_status_table, "maximum_step_uv", right_quality["step_threshold_uv"], "Right-ear maximum-step", "Right step threshold"),
        (axes[1, 0], left_status_table, "peak_to_peak_uv", left_quality["ptp_threshold_uv"], "Left-ear peak-to-peak", "Left PtP threshold"),
        (axes[1, 1], left_status_table, "maximum_step_uv", left_quality["step_threshold_uv"], "Left-ear maximum-step", "Left step threshold"),
    ]

    for ax, table, metric_col, threshold, title, threshold_label in panel_items:
        ax.plot(
            table["start_time_s"],
            table[metric_col],
            color="#c7c7c7",
            linewidth=0.8,
            zorder=1,
        )
        ax.axhline(
            threshold,
            color="#1f77b4",
            linestyle="--",
            linewidth=1.2,
            label=f"{threshold_label} ({threshold:.1f} µV)",
            zorder=2,
        )

        for status_name, style in status_styles.items():
            subset = table.loc[table["status"] == status_name]
            if subset.empty:
                continue
            ax.scatter(
                subset["start_time_s"],
                subset[metric_col],
                c=style["color"],
                marker=style["marker"],
                s=style["size"],
                alpha=style["alpha"],
                label=style["label"],
                zorder=3,
            )

        ax.set_title(title)
        ax.set_ylabel("Amplitude (µV)")
        ax.grid(alpha=0.3)

    axes[1, 0].set_xlabel("Epoch start time after trimming (s)")
    axes[1, 1].set_xlabel("Epoch start time after trimming (s)")

    # Create one combined legend.
    handles, labels = [], []
    for ax in axes.flat:
        h, l = ax.get_legend_handles_labels()
        for hh, ll in zip(h, l):
            if ll not in labels:
                handles.append(hh)
                labels.append(ll)
    fig.legend(
        handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.93),
        ncol=3, frameon=True, fontsize=10,
    )

    summary_text = (
        f"Right kept: {int(right_status_table['final_keep'].sum())}/{len(right_status_table)} | "
        f"Left kept: {int(left_status_table['final_keep'].sum())}/{len(left_status_table)}\n"
        f"Right step-only rejects: {(right_status_table['status'] == 'auto_reject_step').sum()} | "
        f"Left step-only rejects: {(left_status_table['status'] == 'auto_reject_step').sum()}\n"
        f"Manual changes — right: {((right_status_table['status'] == 'manual_bad') | (right_status_table['status'] == 'manual_restored')).sum()} | "
        f"left: {((left_status_table['status'] == 'manual_bad') | (left_status_table['status'] == 'manual_restored')).sum()}"
    )
    fig.text(0.01, 0.01, summary_text, ha="left", va="bottom", fontsize=10)

    fig.suptitle(
        f"{metadata['participant']} - {metadata['scene_name']} - EEG QC (reason-coloured)",
        fontsize=18,
        y=0.98,
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.85])

    output_path = OUTPUT_DIR / f"{file_path.stem}_QC_reason_coloured.png"
    plt.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def process_one_file(file_path: Path):
    metadata = parse_filename(file_path)
    df = load_recording(file_path)

    right_raw_v = df[RIGHT_EEG_COL].to_numpy(dtype=float)
    left_raw_v = df[LEFT_EEG_COL].to_numpy(dtype=float)

    original_duration_s = len(df) / FS

    start_sample = int(round(START_TRIM_SECONDS * FS))
    if END_TRIM_SECONDS > 0:
        end_sample = len(df) - int(round(END_TRIM_SECONDS * FS))
    else:
        end_sample = len(df)

    if start_sample >= end_sample:
        raise ValueError(f"{file_path.name}: trim removes the entire recording.")

    right_analysis_raw_v = right_raw_v[start_sample:end_sample]
    left_analysis_raw_v = left_raw_v[start_sample:end_sample]
    trimmed_duration_s = len(right_analysis_raw_v) / FS

    right_filtered_uv = bandpass_filter(
        right_analysis_raw_v,
        FS,
        EEG_BAND[0],
        EEG_BAND[1],
        FILTER_ORDER,
    ) * V_TO_UV

    left_filtered_uv = bandpass_filter(
        left_analysis_raw_v,
        FS,
        EEG_BAND[0],
        EEG_BAND[1],
        FILTER_ORDER,
    ) * V_TO_UV

    right_epochs = create_epochs(right_filtered_uv, FS, EPOCH_SECONDS)
    left_epochs = create_epochs(left_filtered_uv, FS, EPOCH_SECONDS)

    if right_epochs.shape != left_epochs.shape:
        raise ValueError(f"{file_path.name}: right/left epoch shapes differ.")

    number_of_epochs = right_epochs.shape[0]

    right_quality = assess_epoch_quality(right_epochs, EPOCH_SECONDS, NUMBER_OF_MADS)
    left_quality = assess_epoch_quality(left_epochs, EPOCH_SECONDS, NUMBER_OF_MADS)

    right_status_table = classify_epoch_status(
        right_quality["quality_table"],
        metadata["participant"],
        metadata["scene_code"],
        "right",
        MANUAL_DECISIONS,
    )
    left_status_table = classify_epoch_status(
        left_quality["quality_table"],
        metadata["participant"],
        metadata["scene_code"],
        "left",
        MANUAL_DECISIONS,
    )

    right_raw_epochs_uv = create_epochs(
        right_analysis_raw_v * V_TO_UV, FS, EPOCH_SECONDS
    )
    left_raw_epochs_uv = create_epochs(
        left_analysis_raw_v * V_TO_UV, FS, EPOCH_SECONDS
    )

    for status_table, raw_epochs_uv in [
        (right_status_table, right_raw_epochs_uv),
        (left_status_table, left_raw_epochs_uv),
    ]:
        raw_ptp = np.ptp(raw_epochs_uv, axis=1)
        raw_step = np.max(np.abs(np.diff(raw_epochs_uv, axis=1)), axis=1)
        previous_step = np.concatenate([[np.nan], raw_step[:-1]])
        next_step = np.concatenate([raw_step[1:], [np.nan]])
        neighbour_step = np.nanmedian(
            np.vstack([previous_step, next_step]), axis=0
        )

        status_table["raw_peak_to_peak_uv"] = raw_ptp
        status_table["raw_maximum_step_uv"] = raw_step
        status_table["raw_step_neighbour_median_uv"] = neighbour_step
        status_table["raw_step_ratio_to_neighbours"] = raw_step / neighbour_step

    final_right_clean_mask = right_status_table["final_keep"].to_numpy(dtype=bool)
    final_left_clean_mask = left_status_table["final_keep"].to_numpy(dtype=bool)
    final_joint_clean_mask = final_right_clean_mask & final_left_clean_mask

    clean_epoch_indices = np.flatnonzero(final_joint_clean_mask)
    if len(clean_epoch_indices) == 0:
        raise ValueError(f"{file_path.name}: no jointly clean epochs remain.")

    right_clean_epochs = right_epochs[final_joint_clean_mask]
    left_clean_epochs = left_epochs[final_joint_clean_mask]

    welch_nperseg = int(WELCH_WINDOW_SECONDS * FS)
    welch_noverlap = welch_nperseg // 2

    frequencies, right_psd_epochs = signal.welch(
        right_clean_epochs,
        fs=FS,
        window="hann",
        nperseg=welch_nperseg,
        noverlap=welch_noverlap,
        detrend="constant",
        scaling="density",
        axis=1,
    )

    _, left_psd_epochs = signal.welch(
        left_clean_epochs,
        fs=FS,
        window="hann",
        nperseg=welch_nperseg,
        noverlap=welch_noverlap,
        detrend="constant",
        scaling="density",
        axis=1,
    )

    total_power_mask = (frequencies >= TOTAL_POWER_BAND[0]) & (frequencies <= TOTAL_POWER_BAND[1])
    right_total_power = trapezoid(right_psd_epochs[:, total_power_mask], frequencies[total_power_mask], axis=1)
    left_total_power = trapezoid(left_psd_epochs[:, total_power_mask], frequencies[total_power_mask], axis=1)

    if np.any(right_total_power <= 0) or np.any(left_total_power <= 0):
        raise ValueError(f"{file_path.name}: non-positive 4–30 Hz total power.")

    band_definitions = {
        "theta": THETA_BAND,
        "alpha": ALPHA_BAND,
        "beta": BETA_BAND,
    }

    epoch_result = pd.DataFrame({
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "epoch": clean_epoch_indices,
        "start_time_after_trim_s": clean_epoch_indices * EPOCH_SECONDS,
        "end_time_after_trim_s": (clean_epoch_indices + 1) * EPOCH_SECONDS,
        "right_total_power_4_30_uv2": right_total_power,
        "left_total_power_4_30_uv2": left_total_power,
    })

    for band_name, band_range in band_definitions.items():
        right_power = integrate_band_power(right_psd_epochs, frequencies, band_range[0], band_range[1])
        left_power = integrate_band_power(left_psd_epochs, frequencies, band_range[0], band_range[1])

        epoch_result[f"right_absolute_{band_name}_uv2"] = right_power
        epoch_result[f"left_absolute_{band_name}_uv2"] = left_power
        epoch_result[f"right_relative_{band_name}"] = right_power / right_total_power
        epoch_result[f"left_relative_{band_name}"] = left_power / left_total_power
        epoch_result[f"bilateral_relative_{band_name}"] = (
            epoch_result[f"right_relative_{band_name}"] + epoch_result[f"left_relative_{band_name}"]
        ) / 2
        epoch_result[f"bilateral_absolute_{band_name}_uv2"] = (right_power + left_power) / 2

    epoch_result["bilateral_total_power_4_30_uv2"] = (
        epoch_result["right_total_power_4_30_uv2"] + epoch_result["left_total_power_4_30_uv2"]
    ) / 2

    summary = {
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "clean_epochs": len(epoch_result),
        "usable_duration_s": len(epoch_result) * EPOCH_SECONDS,
        "joint_retention_percent": final_joint_clean_mask.mean() * 100,
    }

    for channel in ["right", "left", "bilateral"]:
        for band in ["theta", "alpha", "beta"]:
            rel_col = f"{channel}_relative_{band}"
            abs_col = f"{channel}_absolute_{band}_uv2"
            summary[f"{channel}_relative_{band}_median"] = epoch_result[rel_col].median()
            if abs_col in epoch_result.columns:
                summary[f"{channel}_absolute_{band}_median_uv2"] = epoch_result[abs_col].median()

    summary["right_total_power_4_30_median_uv2"] = epoch_result["right_total_power_4_30_uv2"].median()
    summary["left_total_power_4_30_median_uv2"] = epoch_result["left_total_power_4_30_uv2"].median()
    summary["bilateral_total_power_4_30_median_uv2"] = epoch_result["bilateral_total_power_4_30_uv2"].median()

    for channel in ["right", "left", "bilateral"]:
        for band in ["theta", "alpha", "beta"]:
            base_col = f"{channel}_relative_{band}_median"
            summary[f"{base_col}_percent"] = summary[base_col] * 100

    def join_epochs(mask_series):
        return ", ".join(map(str, mask_series.astype(int).tolist()))

    def status_count(table, name):
        return int((table["status"] == name).sum())

    qc = {
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "file_name": file_path.name,
        "original_duration_s": original_duration_s,
        "trimmed_duration_s": trimmed_duration_s,
        "total_epochs": number_of_epochs,
        "right_clean_epochs": int(final_right_clean_mask.sum()),
        "left_clean_epochs": int(final_left_clean_mask.sum()),
        "joint_clean_epochs": int(final_joint_clean_mask.sum()),
        "joint_retention_percent": final_joint_clean_mask.mean() * 100,
        "right_ptp_threshold_uv": right_quality["ptp_threshold_uv"],
        "right_step_threshold_uv": right_quality["step_threshold_uv"],
        "left_ptp_threshold_uv": left_quality["ptp_threshold_uv"],
        "left_step_threshold_uv": left_quality["step_threshold_uv"],
        "right_auto_ptp_only_count": status_count(right_status_table, "auto_reject_ptp"),
        "right_auto_step_only_count": status_count(right_status_table, "auto_reject_step"),
        "right_auto_both_count": status_count(right_status_table, "auto_reject_both"),
        "right_manual_bad_count": status_count(right_status_table, "manual_bad"),
        "right_manual_restored_count": status_count(right_status_table, "manual_restored"),
        "left_auto_ptp_only_count": status_count(left_status_table, "auto_reject_ptp"),
        "left_auto_step_only_count": status_count(left_status_table, "auto_reject_step"),
        "left_auto_both_count": status_count(left_status_table, "auto_reject_both"),
        "left_manual_bad_count": status_count(left_status_table, "manual_bad"),
        "left_manual_restored_count": status_count(left_status_table, "manual_restored"),
        "final_rejected_right_epochs": ", ".join(map(str, right_status_table.loc[~right_status_table['final_keep'], 'epoch'].tolist())),
        "final_rejected_left_epochs": ", ".join(map(str, left_status_table.loc[~left_status_table['final_keep'], 'epoch'].tolist())),
        "qc_review_needed": (
            (final_joint_clean_mask.mean() < 0.80)
            or status_count(right_status_table, "auto_reject_step") > 0
            or status_count(left_status_table, "auto_reject_step") > 0
            or status_count(right_status_table, "manual_bad") > 0
            or status_count(left_status_table, "manual_bad") > 0
            or status_count(right_status_table, "manual_restored") > 0
            or status_count(left_status_table, "manual_restored") > 0
        ),
    }

    if SAVE_QC_PLOTS:
        save_qc_plot(
            file_path,
            metadata,
            right_status_table,
            left_status_table,
            right_quality,
            left_quality,
        )

    return summary, qc, epoch_result, right_status_table, left_status_table


## Optional Review Helper


In [4]:
# =========================================================
# OPTIONAL REVIEW HELPER
# =========================================================

def inspect_batch_epoch(participant, scene_code, channel, epoch_index):
    file_path = DATA_DIR / f"{participant}_{scene_code}_EEG.csv"
    metadata = parse_filename(file_path)
    df = load_recording(file_path)

    start_sample = int(round(START_TRIM_SECONDS * FS))
    end_sample = (
        len(df) - int(round(END_TRIM_SECONDS * FS))
        if END_TRIM_SECONDS > 0 else len(df)
    )

    column = RIGHT_EEG_COL if channel == "right" else LEFT_EEG_COL
    raw_v = df[column].to_numpy(dtype=float)[start_sample:end_sample]
    filtered_uv = bandpass_filter(
        raw_v, FS, EEG_BAND[0], EEG_BAND[1], FILTER_ORDER
    ) * V_TO_UV
    epochs = create_epochs(filtered_uv, FS, EPOCH_SECONDS)

    selected = [
        index for index in [epoch_index - 1, epoch_index, epoch_index + 1]
        if 0 <= index < len(epochs)
    ]
    fig, axes = plt.subplots(len(selected) + 1, 1, figsize=(13, 3 * (len(selected) + 1)))

    for ax, current_epoch in zip(axes[:-1], selected):
        epoch_signal = epochs[current_epoch]
        start_s = current_epoch * EPOCH_SECONDS
        time_s = start_s + np.arange(epoch_signal.size) / FS
        ax.plot(time_s, epoch_signal, linewidth=1.2 if current_epoch == epoch_index else 0.9)
        ax.axhline(0, linestyle="--", linewidth=0.7, alpha=0.7)
        ax.set_title(
            f"{'SELECTED' if current_epoch == epoch_index else 'Neighbour'} — "
            f"epoch {current_epoch}: {start_s:.0f}–{start_s + EPOCH_SECONDS:.0f} s | "
            f"PtP = {np.ptp(epoch_signal):.1f} µV | "
            f"Step = {np.max(np.abs(np.diff(epoch_signal))):.1f} µV"
        )
        ax.set_ylabel("Filtered amplitude (µV)")
        ax.grid(alpha=0.3)

    epoch_start_s = epoch_index * EPOCH_SECONDS
    epoch_end_s = epoch_start_s + EPOCH_SECONDS
    context_start_s = max(0, epoch_start_s - 1)
    context_end_s = min(len(raw_v) / FS, epoch_end_s + 1)
    start_index = int(round(context_start_s * FS))
    end_index = int(round(context_end_s * FS))
    raw_time = np.arange(start_index, end_index) / FS
    raw_segment_uv = raw_v[start_index:end_index] * V_TO_UV
    raw_segment_uv -= np.median(raw_segment_uv)

    axes[-1].plot(raw_time, raw_segment_uv, linewidth=0.8)
    axes[-1].axvspan(epoch_start_s, epoch_end_s, alpha=0.15, label=f"Epoch {epoch_index}")
    axes[-1].set_title("Raw signal with one-second context")
    axes[-1].set_xlabel("Time after trimming (s)")
    axes[-1].set_ylabel("Centred raw amplitude (µV)")
    axes[-1].grid(alpha=0.3)
    axes[-1].legend()

    fig.suptitle(
        f"{participant} - {metadata['scene_name']} - {channel}-ear EEG, epoch {epoch_index}",
        fontsize=16, y=0.995,
    )
    plt.tight_layout()
    plt.show()




## 3. Process All Recordings


In [5]:
# =========================================================
# 3. PROCESS ALL RECORDINGS
# =========================================================

if AUTO_DISCOVER_FILES:
    input_files = discover_input_files(DATA_DIR)
else:
    input_files = [DATA_DIR / name for name in EXPECTED_FILES]

print("Files to process (locked pilot set):")
for path in input_files:
    print(" -", path.name)

scene_rows = []
qc_rows = []
epoch_tables = []
status_tables = []
errors = []

for file_path in input_files:
    print(f"\nProcessing: {file_path.name}")
    try:
        scene_row, qc_row, epoch_table, right_status, left_status = process_one_file(file_path)
        scene_rows.append(scene_row)
        qc_rows.append(qc_row)
        epoch_tables.append(epoch_table)

        right_status = right_status.copy()
        right_status["participant"] = scene_row["participant"]
        right_status["scene_code"] = scene_row["scene_code"]
        right_status["scene_name"] = scene_row["scene_name"]
        right_status["channel"] = "right"
        right_status["ptp_threshold_uv"] = qc_row["right_ptp_threshold_uv"]
        right_status["step_threshold_uv"] = qc_row["right_step_threshold_uv"]
        right_status["filtered_ptp_ratio"] = (
            right_status["peak_to_peak_uv"] / right_status["ptp_threshold_uv"]
        )
        right_status["filtered_step_ratio"] = (
            right_status["maximum_step_uv"] / right_status["step_threshold_uv"]
        )
        right_status["review_id"] = (
            right_status["participant"] + "_" + right_status["scene_code"]
            + "_right_epoch" + right_status["epoch"].astype(str)
        )

        left_status = left_status.copy()
        left_status["participant"] = scene_row["participant"]
        left_status["scene_code"] = scene_row["scene_code"]
        left_status["scene_name"] = scene_row["scene_name"]
        left_status["channel"] = "left"
        left_status["ptp_threshold_uv"] = qc_row["left_ptp_threshold_uv"]
        left_status["step_threshold_uv"] = qc_row["left_step_threshold_uv"]
        left_status["filtered_ptp_ratio"] = (
            left_status["peak_to_peak_uv"] / left_status["ptp_threshold_uv"]
        )
        left_status["filtered_step_ratio"] = (
            left_status["maximum_step_uv"] / left_status["step_threshold_uv"]
        )
        left_status["review_id"] = (
            left_status["participant"] + "_" + left_status["scene_code"]
            + "_left_epoch" + left_status["epoch"].astype(str)
        )

        status_tables.extend([right_status, left_status])

        print(
            f"  Done: {scene_row['clean_epochs']} joint clean epochs, "
            f"retention {scene_row['joint_retention_percent']:.1f}%"
        )
    except Exception as exc:
        errors.append({"file_name": file_path.name, "error": str(exc)})
        print(f"  ERROR: {exc}")

scene_eeg_summary_all = pd.DataFrame(scene_rows)
qc_summary_all = pd.DataFrame(qc_rows)
epoch_band_results_all = pd.concat(epoch_tables, ignore_index=True) if epoch_tables else pd.DataFrame()
status_table_all = pd.concat(status_tables, ignore_index=True) if status_tables else pd.DataFrame()

if errors:
    error_table = pd.DataFrame(errors)
    print("\nErrors:")
    print(error_table)
    raise RuntimeError(
        "At least one recording failed. Fix the listed errors before using the combined results."
    )

summary_columns = [
    "participant", "scene_code", "scene_name", "clean_epochs", "usable_duration_s", "joint_retention_percent",
    "right_relative_theta_median", "right_relative_alpha_median", "right_relative_beta_median",
    "left_relative_theta_median", "left_relative_alpha_median", "left_relative_beta_median",
    "bilateral_relative_theta_median", "bilateral_relative_alpha_median", "bilateral_relative_beta_median",
    "right_absolute_theta_median_uv2", "right_absolute_alpha_median_uv2", "right_absolute_beta_median_uv2",
    "left_absolute_theta_median_uv2", "left_absolute_alpha_median_uv2", "left_absolute_beta_median_uv2",
    "bilateral_absolute_theta_median_uv2", "bilateral_absolute_alpha_median_uv2", "bilateral_absolute_beta_median_uv2",
    "right_total_power_4_30_median_uv2", "left_total_power_4_30_median_uv2", "bilateral_total_power_4_30_median_uv2",
    "right_relative_theta_median_percent", "right_relative_alpha_median_percent", "right_relative_beta_median_percent",
    "left_relative_theta_median_percent", "left_relative_alpha_median_percent", "left_relative_beta_median_percent",
    "bilateral_relative_theta_median_percent", "bilateral_relative_alpha_median_percent", "bilateral_relative_beta_median_percent",
]
scene_eeg_summary_all = scene_eeg_summary_all[summary_columns]

# Sort outputs.
def output_sort_key(df):
    return df.sort_values(
        ["participant", "scene_code"],
        key=lambda s: s.map(SCENE_SORT_ORDER) if s.name == "scene_code" else s,
    ).reset_index(drop=True)

scene_eeg_summary_all = output_sort_key(scene_eeg_summary_all)
qc_summary_all = output_sort_key(qc_summary_all)
if not epoch_band_results_all.empty:
    epoch_band_results_all = epoch_band_results_all.sort_values(
        ["participant", "scene_code", "epoch"],
        key=lambda s: s.map(SCENE_SORT_ORDER) if s.name == "scene_code" else s,
    ).reset_index(drop=True)
if not status_table_all.empty:
    status_table_all = status_table_all.sort_values(
        ["participant", "scene_code", "channel", "epoch"],
        key=lambda s: s.map(SCENE_SORT_ORDER) if s.name == "scene_code" else s,
    ).reset_index(drop=True)

file_tag = "pilot8"
scene_csv = OUTPUT_DIR / f"{file_tag}_scene_eeg_summary.csv"
qc_csv = OUTPUT_DIR / f"{file_tag}_qc_summary.csv"
epoch_csv = OUTPUT_DIR / f"{file_tag}_epoch_band_results.csv"
status_csv = OUTPUT_DIR / f"{file_tag}_epoch_qc_status.csv"
review_csv = OUTPUT_DIR / f"{file_tag}_qc_review_candidates.csv"
excel_file = OUTPUT_DIR / f"{file_tag}_EEG_results.xlsx"

scene_eeg_summary_all.to_csv(scene_csv, index=False)
qc_summary_all.to_csv(qc_csv, index=False)
epoch_band_results_all.to_csv(epoch_csv, index=False)
status_table_all.to_csv(status_csv, index=False)

review_candidates = status_table_all.loc[
    status_table_all["auto_reason"] != "kept"
].copy()
review_candidates["review_priority"] = review_candidates["auto_reason"].map(
    {"step_only": "high", "ptp_only": "medium", "both": "low"}
)

decision_columns = [
    "participant", "scene_code", "scene_name", "channel", "epoch",
    "review_id", "review_priority", "auto_reason",
    "peak_to_peak_uv", "maximum_step_uv",
    "filtered_ptp_ratio", "filtered_step_ratio",
    "raw_peak_to_peak_uv", "raw_maximum_step_uv",
    "raw_step_ratio_to_neighbours",
]

base_decision_table = review_candidates[decision_columns].copy()

# Preserve existing decisions and permit manually added rejections of auto-kept epochs.
if DECISION_FILE.exists():
    previous = load_manual_decisions(DECISION_FILE)
else:
    previous = pd.DataFrame(
        columns=[
            "participant", "scene_code", "channel", "epoch",
            "manual_decision", "review_note", "reviewer",
        ]
    )

previous_small = previous[[
    "participant", "scene_code", "channel", "epoch",
    "manual_decision", "review_note", "reviewer",
]].copy()

decision_table = base_decision_table.merge(
    previous_small,
    on=["participant", "scene_code", "channel", "epoch"],
    how="left",
    validate="one_to_one",
)
for column in ["manual_decision", "review_note", "reviewer"]:
    decision_table[column] = decision_table[column].fillna("")

base_keys = set(map(tuple, base_decision_table[[
    "participant", "scene_code", "channel", "epoch"
]].to_numpy()))
previous_keys = list(map(tuple, previous_small[[
    "participant", "scene_code", "channel", "epoch"
]].to_numpy()))
extra_previous = previous_small.loc[[key not in base_keys for key in previous_keys]].copy()

if not extra_previous.empty:
    extra_rows = status_table_all.merge(
        extra_previous,
        on=["participant", "scene_code", "channel", "epoch"],
        how="inner",
        validate="one_to_one",
        suffixes=("_status", ""),
    )
    if len(extra_rows) != len(extra_previous):
        raise ValueError(
            "One or more manually added rows do not match the pilot EEG data."
        )
    extra_rows["review_priority"] = "manual"
    for column in decision_table.columns:
        if column not in extra_rows.columns:
            extra_rows[column] = ""
    decision_table = pd.concat(
        [decision_table, extra_rows[decision_table.columns]],
        ignore_index=True,
    )

decision_table = decision_table.sort_values(
    ["participant", "scene_code", "channel", "epoch"],
    key=lambda s: s.map(SCENE_SORT_ORDER) if s.name == "scene_code" else s,
).reset_index(drop=True)

decision_table.to_csv(DECISION_FILE, index=False)
review_candidates.to_csv(review_csv, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    scene_eeg_summary_all.to_excel(writer, sheet_name="Scene summary", index=False)
    qc_summary_all.to_excel(writer, sheet_name="QC summary", index=False)
    epoch_band_results_all.to_excel(writer, sheet_name="Epoch results", index=False)
    status_table_all.to_excel(writer, sheet_name="Epoch QC status", index=False)
    decision_table.to_excel(writer, sheet_name="Manual decisions", index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nScene summary:")
print(scene_eeg_summary_all.round(4))

print("\nQC rows needing review:")
qc_review_table = qc_summary_all.loc[qc_summary_all["qc_review_needed"]]
if qc_review_table.empty:
    print("None flagged by current review rules.")
else:
    print(qc_review_table)

print("\nSaved files:")
print(scene_csv)
print(qc_csv)
print(epoch_csv)
print(status_csv)
print(review_csv)
print(DECISION_FILE)
print(excel_file)


print("\nSemi-automatic workflow:")
print("1. First run: automatic QC and blank decision file are generated.")
print("2. Edit:", DECISION_FILE)
print("3. Blank = accept automatic result; keep = restore; reject = exclude.")
print("4. Restart the kernel and run all cells again to apply decisions.")


Files to process (locked pilot set):
 - IC01_BG_EEG.csv
 - IC01_G_EEG.csv
 - IC01_GR_EEG.csv
 - IC02_BG_EEG.csv
 - IC02_G_EEG.csv
 - IC02_GR_EEG.csv
 - IC03_BG_EEG.csv
 - IC03_G_EEG.csv
 - IC03_GR_EEG.csv
 - IC04_BG_EEG.csv
 - IC04_G_EEG.csv
 - IC04_GR_EEG.csv
 - IC05_BG_EEG.csv
 - IC05_G_EEG.csv
 - IC05_GR_EEG.csv
 - N01_BG_EEG.csv
 - N01_G_EEG.csv
 - N01_GR_EEG.csv
 - N02_BG_EEG.csv
 - N02_G_EEG.csv
 - N02_GR_EEG.csv
 - N03_BG_EEG.csv
 - N03_G_EEG.csv
 - N03_GR_EEG.csv

Processing: IC01_BG_EEG.csv
  Done: 51 joint clean epochs, retention 73.9%

Processing: IC01_G_EEG.csv
  Done: 69 joint clean epochs, retention 97.2%

Processing: IC01_GR_EEG.csv
  Done: 71 joint clean epochs, retention 100.0%

Processing: IC02_BG_EEG.csv
  Done: 66 joint clean epochs, retention 95.7%

Processing: IC02_G_EEG.csv
  Done: 70 joint clean epochs, retention 100.0%

Processing: IC02_GR_EEG.csv
  Done: 67 joint clean epochs, retention 94.4%

Processing: IC03_BG_EEG.csv
  Done: 69 joint clean epochs, retentio